# Задание Pro — PyTorch

Используя датасет **«Пассажиры автобуса»**, создайте нейронную сеть для классификации пассажиров на входящих и выходящих.

Цель: получить точность выше 90% на проверочной/тестовой выборке.

In [ ]:
# Загрузка библиотек

import os
import zipfile
import shutil
import random
import time

import gdown
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Фиксация случайности для повторяемости результата
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)

In [ ]:
# Загрузка и распаковка датасета

DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l4/bus.zip'
ZIP_PATH = '/content/bus.zip'
IMAGE_PATH = '/content/bus/'

# Скачиваем архив только при отсутствии файла
if not os.path.exists(ZIP_PATH):
    gdown.download(DATA_URL, ZIP_PATH, quiet=True)

# Очищаем старую папку, чтобы повторный запуск не создавал мусор
if os.path.exists(IMAGE_PATH):
    shutil.rmtree(IMAGE_PATH)

os.makedirs(IMAGE_PATH, exist_ok=True)

# Распаковка архива без команды !unzip
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    archive.extractall(IMAGE_PATH)

print('Папки в датасете:', os.listdir(IMAGE_PATH))

In [ ]:
# Определение классов

CLASS_LIST = sorted([
    name for name in os.listdir(IMAGE_PATH)
    if os.path.isdir(os.path.join(IMAGE_PATH, name)) and not name.startswith('.')
])

CLASS_COUNT = len(CLASS_LIST)

print(f'Количество классов: {CLASS_COUNT}')
print('Метки классов:', CLASS_LIST)

for cls in CLASS_LIST:
    class_path = os.path.join(IMAGE_PATH, cls)
    files = [
        file_name for file_name in os.listdir(class_path)
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]
    print(f'{cls}: {len(files)} изображений')

In [ ]:
# Просмотр случайных изображений из каждого класса

fig, axs = plt.subplots(1, CLASS_COUNT, figsize=(10, 5))
axs = np.atleast_1d(axs)

for i, class_name in enumerate(CLASS_LIST):
    class_path = os.path.join(IMAGE_PATH, class_name)

    image_names = [
        file_name for file_name in os.listdir(class_path)
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ]

    img_path = os.path.join(class_path, random.choice(image_names))
    img = Image.open(img_path).convert('L')

    axs[i].imshow(img, cmap='gray')
    axs[i].set_title(class_name)
    axs[i].axis('off')

plt.show()

In [ ]:
# Формирование списков файлов и меток

data_files = []
data_labels = []
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

for class_label, class_name in enumerate(CLASS_LIST):
    class_path = os.path.join(IMAGE_PATH, class_name)

    class_files = [
        file_name for file_name in os.listdir(class_path)
        if file_name.lower().endswith(valid_extensions)
    ]

    print(f'Размер класса {class_name}: {len(class_files)} изображений')

    for file_name in class_files:
        data_files.append(os.path.join(class_path, file_name))
        data_labels.append(class_label)

if len(data_files) == 0:
    raise ValueError('Изображения не найдены. Проверьте структуру датасета.')

print('Всего изображений:', len(data_files))
print('Пример меток:', data_labels[:10])

In [ ]:
class BusPassengerCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3)
        )

        # Исходный размер: 120x80.
        # После трех MaxPool2d(2): 15x10.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 15 * 10, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, class_count)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class PassengerTorch:
    def __init__(self, epochs=18, batch_size=8, learning_rate=0.001):
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.img_height = 120
        self.img_width = 80
        self.model = None
        self.history = None
        self.class_count = CLASS_COUNT

    # Загрузка одного изображения
    def load_image(self, file_path):
        img = Image.open(file_path).convert('L')
        img = img.resize((self.img_width, self.img_height))

        img_array = np.array(img, dtype=np.float32) / 255.0

        # PyTorch ждёт формат: канал, высота, ширина
        img_array = np.expand_dims(img_array, axis=0)

        return img_array

    # Загрузка всего датасета
    def load_dataset(self):
        x_data = []
        y_data = []

        for file_path, label in zip(data_files, data_labels):
            try:
                x_data.append(self.load_image(file_path))
                y_data.append(label)
            except Exception as error:
                print(f'Файл пропущен: {file_path}. Причина: {error}')

        if len(x_data) == 0:
            raise ValueError('Изображения не загружены. Проверьте путь к датасету.')

        x_data = np.array(x_data, dtype=np.float32)
        y_data = np.array(y_data, dtype=np.int64)

        return x_data, y_data

    # Подготовка выборок
    def load_data(self):
        x_data, y_data = self.load_dataset()

        x_train_val, x_test, y_train_val, y_test = train_test_split(
            x_data,
            y_data,
            test_size=0.2,
            random_state=SEED,
            stratify=y_data
        )

        x_train, x_val, y_train, y_val = train_test_split(
            x_train_val,
            y_train_val,
            test_size=0.25,
            random_state=SEED,
            stratify=y_train_val
        )

        train_dataset = TensorDataset(
            torch.tensor(x_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long)
        )

        val_dataset = TensorDataset(
            torch.tensor(x_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long)
        )

        test_dataset = TensorDataset(
            torch.tensor(x_test, dtype=torch.float32),
            torch.tensor(y_test, dtype=torch.long)
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

        return train_loader, val_loader, test_loader

    # Создание модели
    def create_model(self):
        return BusPassengerCNN(self.class_count).to(device)

    # Один проход обучения
    def train_one_epoch(self, model, loader, loss_function, optimizer):
        model.train()

        total_loss = 0
        correct = 0
        total = 0

        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(x_batch)
            loss = loss_function(outputs, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x_batch.size(0)

            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

        avg_loss = total_loss / total
        avg_accuracy = correct / total

        return avg_loss, avg_accuracy

    # Проверка модели
    def evaluate_model(self, model, loader, loss_function):
        model.eval()

        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for x_batch, y_batch in loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(x_batch)
                loss = loss_function(outputs, y_batch)

                total_loss += loss.item() * x_batch.size(0)

                predicted = torch.argmax(outputs, dim=1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        avg_loss = total_loss / total
        avg_accuracy = correct / total

        return avg_loss, avg_accuracy

    # Обучение модели
    def train_model(self):
        train_loader, val_loader, test_loader = self.load_data()

        model = self.create_model()
        loss_function = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=self.learning_rate)

        history = {
            'loss': [],
            'accuracy': [],
            'val_loss': [],
            'val_accuracy': []
        }

        best_val_accuracy = 0
        best_model_path = '/content/best_bus_passenger_model.pth'

        start_time = time.time()

        for epoch in range(1, self.epochs + 1):
            train_loss, train_accuracy = self.train_one_epoch(
                model,
                train_loader,
                loss_function,
                optimizer
            )

            val_loss, val_accuracy = self.evaluate_model(
                model,
                val_loader,
                loss_function
            )

            history['loss'].append(train_loss)
            history['accuracy'].append(train_accuracy)
            history['val_loss'].append(val_loss)
            history['val_accuracy'].append(val_accuracy)

            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                torch.save(model.state_dict(), best_model_path)

            print(
                f'Эпоха {epoch:02d}/{self.epochs} | '
                f'ошибка: {train_loss:.4f} | '
                f'точность: {train_accuracy:.4f} | '
                f'проверочная ошибка: {val_loss:.4f} | '
                f'проверочная точность: {val_accuracy:.4f}'
            )

        # Загружаем лучшую модель по проверочной точности
        model.load_state_dict(torch.load(best_model_path, map_location=device))

        test_loss, test_accuracy = self.evaluate_model(
            model,
            test_loader,
            loss_function
        )

        self.model = model
        self.history = history

        elapsed_time = time.time() - start_time

        print(f'\nТочность на тестовой выборке: {test_accuracy * 100:.2f}%')
        print(f'Ошибка на тестовой выборке: {test_loss:.4f}')
        print(f'Лучшая проверочная точность: {best_val_accuracy * 100:.2f}%')
        print(f'Время обучения: {elapsed_time:.1f} сек.')
        print(f'Модель сохранена: {best_model_path}')

        return model, history, test_loss, test_accuracy

    # Графики
    def show_graphs(self, history=None):
        if history is None:
            history = self.history

        if history is None:
            raise ValueError('История обучения отсутствует. Сначала выполните train_model().')

        epochs = range(1, len(history['accuracy']) + 1)

        plt.figure(figsize=(10, 5))
        plt.plot(epochs, history['accuracy'], label='Точность на обучающей выборке')
        plt.plot(epochs, history['val_accuracy'], label='Точность на проверочной выборке')
        plt.xlabel('Эпоха')
        plt.ylabel('Точность')
        plt.title('График точности модели PyTorch')
        plt.legend()
        plt.grid()
        plt.show()

        plt.figure(figsize=(10, 5))
        plt.plot(epochs, history['loss'], label='Ошибка на обучающей выборке')
        plt.plot(epochs, history['val_loss'], label='Ошибка на проверочной выборке')
        plt.xlabel('Эпоха')
        plt.ylabel('Ошибка')
        plt.title('График ошибки модели PyTorch')
        plt.legend()
        plt.grid()
        plt.show()

    # Предсказание по одному изображению
    def predict_image(self, file_path):
        if self.model is None:
            raise ValueError('Модель не обучена. Сначала выполните train_model().')

        self.model.eval()

        img_array = self.load_image(file_path)
        x = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = self.model(x)
            probabilities = torch.softmax(outputs, dim=1)
            predicted_index = torch.argmax(probabilities, dim=1).item()

        predicted_class = CLASS_LIST[predicted_index]
        confidence = probabilities[0, predicted_index].item()

        print(f'Класс: {predicted_class}')
        print(f'Уверенность: {confidence * 100:.2f}%')

        return predicted_class, confidence


In [ ]:
# Создание и обучение модели

passenger = PassengerTorch(epochs=18, batch_size=8, learning_rate=0.001)
model, history, loss, accuracy = passenger.train_model()

print(model)

In [ ]:
# Вывод графиков

passenger.show_graphs(history)

## Примечание

В PyTorch для классификации используется `CrossEntropyLoss`. Поэтому последний слой модели не содержит `Softmax`: функция потерь сама корректно обрабатывает выходные значения модели.